# Find 2025's unusual traffic days

*Data Analysis in Python · Topic 8.2*

The analytics team is setting traffic alerts for 2026 and wants to know what a normal day looked like in 2025, and which days were unusual enough to set one off.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

1. Run the parameters cell, the cell that builds the `sessions` table (lesson 3.'s steps) and the cell that builds the `daily` table: one row per day, with the dates as the index (topic 6.1.'s steps).
2. In the first cell marked ✏️, create `typical_day`: the median of the `daily` table's `sessions`.
3. In the second cell marked ✏️, create, from the `daily` table's `sessions`:
   - `q1` and `q3`: the 25th and 75th percentiles
   - `lower_fence` and `upper_fence`: the IQR rule's fences, 1.5 IQRs below `q1` and above `q3`
   - the `outlier_days` table: the rows of the `daily` table whose sessions are above `upper_fence`
4. In the last cell marked ✏️, create `hist_ax`: the Axes of a histogram of the `daily` table's `sessions`, with a vertical line at `upper_fence`, a title and an x-axis label.
5. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: `typical_day` is `3614.0`. `q1` is `3224.0`, `q3` `4304.0`, `lower_fence` `1604.0` and `upper_fence` `5924.0`. The `outlier_days` table has 13 days, from November 21 to December 9, 2025, led by Black Friday, November 28, with 14,786 sessions. `hist_ax` has a histogram of the 365 days, a vertical line at the upper fence, a title and an x-axis label.

## Parameters

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = [f"2025-{month:02d}" for month in range(1, 13)]
START, END = "2025-01-01", "2025-12-31"

In [ ]:
# The clean sessions table, with lesson 3's steps. Run it, and leave it as it is.
DAY_FIRST_MONTHS = ["2025-03"]
SOURCE_ALIASES = {
    "facebook.com": "facebook", "m.facebook.com": "facebook", "l.facebook.com": "facebook",
    "lm.facebook.com": "facebook", "fb": "facebook",
    "instagram.com": "instagram", "l.instagram.com": "instagram", "ig": "instagram",
}


def read_export(month):
    """Read one month's export with dates as dates, revenue as numbers and lost campaigns as missing."""
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    return export


raw = pd.concat([read_export(month) for month in MONTHS], ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["source"] = sessions["session_source"].str.lower().str.strip().replace(SOURCE_ALIASES)
sessions.shape

In [ ]:
# One row per day, with the dates as the index (topic 6.1). Run it, and leave it as it is.
daily = sessions.groupby("date")[["sessions", "transactions", "revenue"]].sum()
daily.shape

## A typical day

In [ ]:
typical_day = daily["sessions"].median()
typical_day

## The fences

In [ ]:
q1, q3 = daily["sessions"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outlier_days = daily[daily["sessions"] > upper_fence]
lower_fence, upper_fence, len(outlier_days)

## The chart

In [ ]:
fig, hist_ax = plt.subplots(figsize=(10, 4))
hist_ax.hist(daily["sessions"], bins=30)
hist_ax.axvline(upper_fence, color="red", linestyle="--", label=f"Outlier fence: {upper_fence:,.0f}")
hist_ax.set_title("Daily sessions, 2025")
hist_ax.set_xlabel("Sessions per day")
hist_ax.set_ylabel("Days")
hist_ax.legend()
plt.show()

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables, your charts and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import matplotlib.axes
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    else:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
        if checked_before:
            results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
        elif repeated:
            first_line = repeated[0].strip().splitlines()[0]
            results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
        elif error is not None:
            results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
        else:
            results.append((True, "Every cell in this session ran once, with no errors."))

    def number(value):
        return pd.api.types.is_number(value) and not isinstance(value, (pd.DataFrame, pd.Series))

    typical = found.get("typical_day")
    if not number(typical):
        results.append((False, f"typical_day should be a number: the median of the daily table's sessions, daily[\"sessions\"].median(). It is {typical!r}."))
    elif abs(typical - 3922.51) < 0.01:
        results.append((False, "typical_day is 3,922.51: that's the mean. The 13 Black Friday season days pull it up. A typical day is the median: daily[\"sessions\"].median()."))
    elif typical != 3614:
        results.append((False, f"typical_day should be the median of the daily table's sessions, 3,614. It is {typical!r}."))
    else:
        results.append((True, "A typical day in 2025 had 3,614 sessions: the median, which the busiest days don't pull up the way they pull the mean, 3,922.51."))

    q1, q3 = found.get("q1"), found.get("q3")
    lower, upper = found.get("lower_fence"), found.get("upper_fence")
    outliers = found.get("outlier_days")
    if not all(number(value) for value in (q1, q3, lower, upper)):
        results.append((False, "q1, q3, lower_fence and upper_fence should be numbers: q1, q3 = daily[\"sessions\"].quantile([0.25, 0.75]), then the fences from them."))
    elif (q1, q3) == (4304, 3224):
        results.append((False, "q1 and q3 are swapped: q1 is 4,304 and q3 3,224. quantile([0.25, 0.75]) gives the 25th percentile first."))
    elif (q1, q3) == (5, 65) or abs(upper - 155) < 0.01:
        results.append((False, "The fences come from the sessions table's rows, one per source, medium and campaign. The question is about days: take the quartiles of the daily table's sessions."))
    elif q1 != 3224 or q3 != 4304:
        results.append((False, f"q1 and q3 should be the 25th and 75th percentiles of the daily table's sessions, 3,224 and 4,304. They are {q1!r} and {q3!r}."))
    elif abs(upper - 7678.14) < 0.01 or abs(upper - 6426.26) < 0.01:
        results.append((False, f"upper_fence is {upper:,.2f}: that's the mean plus {'three' if upper > 7000 else 'two'} standard deviations. The brief asks for the IQR rule: q3 + 1.5 * (q3 - q1)."))
    elif abs(upper - 7544) < 0.01:
        results.append((False, "upper_fence is 7,544: that's 3 IQRs above q3. The IQR rule, and the box plot, use 1.5: q3 + 1.5 * (q3 - q1)."))
    elif abs(upper - 5924) > 0.01 or abs(lower - 1604) > 0.01:
        results.append((False, f"The fences should be 1.5 IQRs beyond the quartiles: lower_fence 1,604 and upper_fence 5,924. They are {lower!r} and {upper!r}."))
    elif not isinstance(outliers, (pd.DataFrame, pd.Series)):
        results.append((False, "outlier_days should be the daily table's rows above the fence: daily[daily[\"sessions\"] > upper_fence]."))
    elif len(outliers) == 38:
        results.append((False, "The outlier_days table has 38 days: those are revenue's outliers. Filter on the daily table's sessions."))
    elif len(outliers) != 13 or not isinstance(outliers.index, pd.DatetimeIndex):
        results.append((False, f"The outlier_days table should have 13 days, with their dates as the index. It has {len(outliers)} rows."))
    else:
        top = outliers["sessions"] if isinstance(outliers, pd.DataFrame) else outliers
        results.append((True, f"13 days above the fence of 5,924 sessions, from {outliers.index.min():%B %-d} to {outliers.index.max():%B %-d}, led by {top.idxmax():%B %-d} with {top.max():,}. None below 1,604."))

    hist_ax = found.get("hist_ax")
    if not isinstance(hist_ax, matplotlib.axes.Axes):
        results.append((False, f"hist_ax should be the Axes of your histogram: fig, hist_ax = plt.subplots(), then hist_ax.hist(daily[\"sessions\"], bins=30). It is a {type(hist_ax).__name__}."))
    else:
        counts = [patch.get_height() for patch in hist_ax.patches if patch.get_width()]
        fence_lines = [line for line in hist_ax.get_lines() if len(set(line.get_xdata())) == 1 and number(line.get_xdata()[0]) and abs(line.get_xdata()[0] - 5924) < 0.5]
        if not counts:
            results.append((False, "hist_ax has no bars. Draw the histogram on it: hist_ax.hist(daily[\"sessions\"], bins=30)."))
        elif sum(counts) == 21880:
            results.append((False, "hist_ax's histogram counts 21,880 values: the sessions table's rows. Draw the daily table's sessions, one value per day."))
        elif sum(counts) != 365:
            results.append((False, f"hist_ax's bars should count the 365 days of 2025. They count {sum(counts):,.0f}."))
        elif not fence_lines:
            results.append((False, "hist_ax has no vertical line at upper_fence. Add one: hist_ax.axvline(upper_fence, color=\"red\", linestyle=\"--\")."))
        elif not hist_ax.get_title().strip():
            results.append((False, "hist_ax has no title. Say what and when: hist_ax.set_title(\"Daily sessions, 2025\")."))
        elif not hist_ax.get_xlabel().strip():
            results.append((False, "hist_ax has no x-axis label. Say what the values are: hist_ax.set_xlabel(\"Sessions per day\")."))
        else:
            results.append((True, f"The histogram counts the 365 days in {len(counts)} bins, with the fence at 5,924 sessions, and it's titled {hist_ax.get_title()!r}."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
typical_day = daily["sessions"].median()
typical_day
```

```python
q1, q3 = daily["sessions"].quantile([0.25, 0.75])
iqr = q3 - q1
lower_fence = q1 - 1.5 * iqr
upper_fence = q3 + 1.5 * iqr
outlier_days = daily[daily["sessions"] > upper_fence]
lower_fence, upper_fence, len(outlier_days)
```

```python
fig, hist_ax = plt.subplots(figsize=(10, 4))
hist_ax.hist(daily["sessions"], bins=30)
hist_ax.axvline(upper_fence, color="red", linestyle="--", label=f"Outlier fence: {upper_fence:,.0f}")
hist_ax.set_title("Daily sessions, 2025")
hist_ax.set_xlabel("Sessions per day")
hist_ax.set_ylabel("Days")
hist_ax.legend()
plt.show()
```

</details>